In [ ]:
# =========================
# LAB 10 - GENERATIVE AI
# =========================

import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, LayerNormalization, MultiHeadAttention

# =========================
# DATASET
# =========================
data = """artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.

deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.

machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.

technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making."""

# =========================
# TOKENIZATION
# =========================
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

# =========================
# CREATE SEQUENCES
# =========================
input_sequences = []

for line in data.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# =========================
# PADDING
# =========================
max_seq_len = max(len(x) for x in input_sequences)

input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# =========================
# LSTM MODEL
# =========================
print("\nTraining LSTM Model...\n")

model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=1)

# =========================
# LSTM TEXT GENERATION
# =========================
def generate_text(seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        
        predicted = np.argmax(model.predict(token_list), axis=-1)[0]
        
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

print("\nLSTM Output:")
print(generate_text("machine learning", 5))


# =========================
# TRANSFORMER MODEL
# =========================
print("\nTraining Transformer Model...\n")

def transformer_block(x):
    attn = MultiHeadAttention(num_heads=2, key_dim=64)(x, x)
    x = LayerNormalization()(x + attn)
    
    ffn = Dense(128, activation='relu')(x)
    ffn = Dense(64)(ffn)
    
    x = LayerNormalization()(x + ffn)
    return x

inputs = Input(shape=(max_seq_len-1,))
x = Embedding(total_words, 64)(inputs)

x = transformer_block(x)

x = Dense(64, activation='relu')(x)
x = Dense(total_words, activation='softmax')(x[:, -1, :])

transformer_model = Model(inputs, x)

transformer_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
transformer_model.fit(X, y, epochs=100, verbose=1)

# =========================
# TRANSFORMER TEXT GENERATION
# =========================
def generate_text_transformer(seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        
        predicted = np.argmax(transformer_model.predict(token_list), axis=-1)[0]
        
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

print("\nTransformer Output:")
print(generate_text_transformer("deep learning", 5))

print("\n✅ LAB 10 COMPLETED SUCCESSFULLY")